# DS675 Mini-Project: Diabetes Health Indicators
## KNN - Hyperparameter tunning and feature tunning




In [1]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             precision_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

## 0. Setup — load file.
### This is a binary, balanced version of the dataset

In [2]:
#load file from drive.
from google.colab import drive
import os

# 1. Mount your Google Drive
# You will be prompted to grant permission the first time you run this
drive.mount('/content/drive')

# 2. Define the path (Option A: MyDrive with no space)
# Note: Ensure "Colab Notebooks" matches the exact spelling/capitalization in your Drive
csv_path = "/content/drive/MyDrive/Colab Notebooks/DS675/mini_project/data/diabetes_binary_5050split_health_indicators_BRFSS2015.csv"

# 3. Verify if the file exists at that path
if os.path.exists(csv_path):
    print("Success! File found.")
    # 4. Read the CSV
    df_full = pd.read_csv(csv_path)

    # Display the first 5 rows to confirm data loaded correctly
    #print(df.head())
else:
    print("File not found at the specified path.")
    print("Check your folder names for typos or try 'My Drive' with a space.")


Mounted at /content/drive
Success! File found.


##1. Train and Scale

In [6]:
# split, scale
X = df_full.drop('Diabetes_binary', axis=1)
y = df_full['Diabetes_binary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_std = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_std = pd.DataFrame(X_test_scaled, columns=X.columns)

## 2. Hyperparameter tuning for KNN

In [5]:
# Hyperparameter tuning for knn
knn = KNeighborsClassifier()
param_grid = {
    'n_neighbors': np.arange(3, 50, 5),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}
# cv=5 means 5-fold cross-validation
grid_KNN = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')

# 3. Fit to the training data
# Remember to use the SCALED data from the previous step
grid_KNN.fit(X_train_std, y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'metric': ['euclidean', 'manhattan'],
                         'n_neighbors': array([ 3,  8, 13, 18, 23, 28, 33, 38, 43, 48]),
                         'weights': ['uniform', 'distance']},
             scoring='accuracy')

## 3. Results for the best KNN
#### This is the best result found for KNN for our dataset

In [7]:
########################
#Read results
########################
results = pd.DataFrame(grid_KNN.cv_results_)

# Filtering to the most important columns for readability
relevant_columns = ['param_n_neighbors', 'param_weights', 'param_metric', 'mean_test_score', 'rank_test_score']
all_results = results[relevant_columns].sort_values(by='rank_test_score')

print("--- Top 10 Parameter Combinations ---")
print(all_results.head(10))

print(f"\nBest Parameters: {grid_KNN.best_params_}")
print(f"Best Cross-Validation Score: {grid_KNN.best_score_:.4f}")

# 5. Use the best model to predict
best_knn = grid_KNN.best_estimator_
final_predictions = best_knn.predict(X_test_std)

# 6. Evaluate

print(f"Accuracy: {accuracy_score(y_test, final_predictions):.2%}")
classificationreport = classification_report(y_test, final_predictions)
print(classificationreport)

--- Top 10 Parameter Combinations ---
    param_n_neighbors param_weights param_metric  mean_test_score  \
36                 43       uniform    manhattan         0.743161   
38                 48       uniform    manhattan         0.743126   
34                 38       uniform    manhattan         0.741534   
32                 33       uniform    manhattan         0.740951   
39                 48      distance    manhattan         0.740880   
37                 43      distance    manhattan         0.740774   
35                 38      distance    manhattan         0.740244   
16                 43       uniform    euclidean         0.739554   
18                 48       uniform    euclidean         0.739395   
30                 28       uniform    manhattan         0.739359   

    rank_test_score  
36                1  
38                2  
34                3  
32                4  
39                5  
37                6  
35                7  
16                8  
18  

##4. Create datasets with 1 - 21 features.
####ordered by the importance given in SHAP analysis

In [8]:
# Shap chose list of features
sorted_features = ['GenHlth', 'HighBP', 'BMI', 'Age', 'HighChol', 'Sex', 'Income', 'HeartDiseaseorAttack',
                   'DiffWalk', 'MentHlth', 'CholCheck', 'HvyAlcoholConsump', 'Education', 'PhysHlth', 'Stroke',
                   'Smoker', 'PhysActivity', 'Fruits', 'Veggies', 'AnyHealthcare', 'NoDocbcCost']
#####################################
# create data frames with k features
#####################################
X_trains = list(range(22))
for i in range(1, 22):
    X_trains[i] = X_train_std[sorted_features[0:i]].copy()

X_tests = list(range(22))
for i in range(1, 22):
    X_tests[i] = X_test_std[sorted_features[0:i]].copy()

##5. Fit the best KNN to each of the partial models

In [9]:
from sklearn.base import clone
######################
#feature counting
#Explore the number of features needed for maximal accurracy with KNN
######################

#########################################################
################# Knn - multipule ks ####################
#########################################################

#keep scores
accuracy_scores = list(range(22))

# fit the best modals to the partial datasets
for i in range(1, 22):
  # create a clone to make sure no data from last training attaches
  best_knn_p = clone(best_knn)
  best_knn_p.fit(X_trains[i], y_train)
  final_predictions = best_knn_p.predict(X_tests[i])

  #Results
  print("\n"+"*"*50)
  print(f"Number of features: {i}")
  accuracy_scores[i] = accuracy_score(y_test, final_predictions)
  print(f"Accuracy: {accuracy_score(y_test, final_predictions):.4%}")
  classificationreport = classification_report(y_test, final_predictions)
  print('\nClassification_report : ')
  print(classificationreport)
  print("\n"+"*"*50)



**************************************************
Number of features: 1
Accuracy: 67.9327%

Classification_report : 
              precision    recall  f1-score   support

         0.0       0.73      0.57      0.64      7070
         1.0       0.65      0.79      0.71      7069

    accuracy                           0.68     14139
   macro avg       0.69      0.68      0.68     14139
weighted avg       0.69      0.68      0.68     14139


**************************************************

**************************************************
Number of features: 2
Accuracy: 70.6203%

Classification_report : 
              precision    recall  f1-score   support

         0.0       0.70      0.72      0.71      7070
         1.0       0.71      0.69      0.70      7069

    accuracy                           0.71     14139
   macro avg       0.71      0.71      0.71     14139
weighted avg       0.71      0.71      0.71     14139


**************************************************

***

##6. How fast does the partial sets converge to the full set?

In [12]:
#Find when we are preety close to the maximum accurecy
full_acc = accuracy_scores[-1]
found80 = found90 = found95 = False
acc_80 = acc_90 = acc_95 = 21

i = 1
while i < 22 and (not found80 or not found90 or not found95):
  if (accuracy_scores[i] / full_acc) > 0.8 and not found80:
    acc_80 = i
    found80 = True
  if (accuracy_scores[i] / full_acc) > 0.9 and not found90:
    acc_90 = i
    found90 = True
  if (accuracy_scores[i] / full_acc) > 0.95 and not found95:
    acc_95 = i
    found95 = True
  i = i+1

print(f"80% Number of features: {acc_80}")
print(f"Accuracy: {accuracy_scores[acc_80] / full_acc:.4%}")

print(f"90% Number of features: {acc_90}")
print(f"Accuracy: {accuracy_scores[acc_90] / full_acc:.4%}")

print(f"95% Number of features: {acc_95}")
print(f"Accuracy: {accuracy_scores[acc_95] / full_acc:.4%}")


80% Number of features: 1
Accuracy: 91.7383%
90% Number of features: 1
Accuracy: 91.7383%
95% Number of features: 2
Accuracy: 95.3677%


## 7. Conclusion

In [13]:
features = sorted_features[0:acc_95]
print(f"KNN has {accuracy_scores[-1]:.2%} accuracy when considering all features." )
print(f"When considering just the features  { {*features} }  the accuracy is {accuracy_scores[acc_95]:.2%}")
print(f"which is {accuracy_scores[acc_95] / full_acc:.2%}  of the whole set accuracy")

KNN has 74.05% accuracy when considering all features.
When considering just the features  {'GenHlth', 'HighBP'}  the accuracy is 70.62%
which is 95.37%  of the whole set accuracy


In [2]:
!jupyter nbconvert --to html "/content/drive/MyDrive/Colab Notebooks/DS675/mini_project/KNNTuning.html"


[NbConvertApp] WARNING | pattern '/content/drive/MyDrive/Colab Notebooks/DS675/mini_project/KNNTuning.html' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equi